# Credit Card Fraud Detection

This notebook walks through the complete workflow for detecting fraudulent credit card
transactions: data cleaning, exploratory data analysis, handling class imbalance with
SMOTE, feature engineering, model training and comparison, and fraud pattern analysis.

All paths below are anchored to the project folder so the notebook runs unchanged from
any location.

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, roc_curve
)

PROJECT_ROOT = os.path.dirname(os.getcwd())
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
CHARTS_DIR = os.path.join(PROJECT_ROOT, "charts")

sns.set_style("whitegrid")
RANDOM_STATE = 42

## 1. Data Cleaning & Preprocessing

In [ ]:
df_raw = pd.read_csv(os.path.join(DATA_DIR, "raw_transactions.csv"))
print(df_raw.shape)
df_raw.head()

In [ ]:
print("Missing values:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print("\nDuplicate rows:", df_raw.duplicated().sum())

In [ ]:
df = df_raw.drop_duplicates()
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
df = df[df["Amount"] >= 0].reset_index(drop=True)
df["Class"] = df["Class"].astype(int)
print("Cleaned shape:", df.shape)

## 2. Exploratory Data Analysis

In [ ]:
class_counts = df["Class"].value_counts()
print(class_counts)
print(f"Fraud rate: {df['Class'].mean() * 100:.3f}%")

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Legitimate", "Fraud"], class_counts.values, color=["#2E86AB", "#E63946"])
ax.set_title("Class Distribution")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, name, color in [(0, "Legitimate", "#2E86AB"), (1, "Fraud", "#E63946")]:
    subset = df.loc[df["Class"] == label, "Amount"]
    ax.hist(subset.clip(upper=subset.quantile(0.99)), bins=40, alpha=0.6,
            label=name, color=color, density=True)
ax.set_title("Transaction Amount Distribution by Class")
ax.legend()
plt.show()

In [ ]:
v_cols = [c for c in df.columns if c.startswith("V")]
corr_with_class = df[v_cols + ["Class"]].corr()["Class"].drop("Class").sort_values()
corr_with_class.tail(10)

## 3. Feature Engineering

In [ ]:
df["Hour"] = (df["Time"] % (24 * 3600)) // 3600
df["Day"] = (df["Time"] // (24 * 3600)).astype(int)
df["Amount_log"] = np.log1p(df["Amount"])

feature_cols = [c for c in df.columns if c != "Class"]
X = df[feature_cols].values
y = df["Class"].values
X.shape, y.shape

## 4. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Train fraud rate: {y_train.mean()*100:.3f}%  Test fraud rate: {y_test.mean()*100:.3f}%")

## 5. Handling Class Imbalance (SMOTE)

A k-NN based SMOTE implementation is used to synthetically oversample the minority
(fraud) class in the training set only, so the model sees a balanced training
distribution while the test set keeps its original, realistic imbalance.

In [ ]:
def smote_oversample(X, y, minority_label=1, k_neighbors=5, random_state=RANDOM_STATE):
    rng = np.random.default_rng(random_state)
    X_minority = X[y == minority_label]
    n_minority = len(X_minority)
    n_majority = (y != minority_label).sum()
    n_to_generate = n_majority - n_minority
    if n_to_generate <= 0:
        return X, y

    k = min(k_neighbors, n_minority - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_minority)
    _, neighbor_idx = nn.kneighbors(X_minority)

    synthetic_samples = []
    for _ in range(n_to_generate):
        i = rng.integers(0, n_minority)
        j = neighbor_idx[i, rng.integers(1, k + 1)]
        gap = rng.random()
        synthetic_samples.append(X_minority[i] + gap * (X_minority[j] - X_minority[i]))

    X_synthetic = np.vstack(synthetic_samples)
    y_synthetic = np.full(n_to_generate, minority_label)
    X_res = np.vstack([X, X_synthetic])
    y_res = np.concatenate([y, y_synthetic])
    shuffle_idx = rng.permutation(len(y_res))
    return X_res[shuffle_idx], y_res[shuffle_idx]

X_train_bal, y_train_bal = smote_oversample(X_train_scaled, y_train)
print("Balanced training set:", X_train_bal.shape, f"fraud share: {y_train_bal.mean()*100:.1f}%")

## 6. Train & Compare Classification Models

In [ ]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = {
        "precision": round(precision_score(y_test, y_pred), 4),
        "recall": round(recall_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred), 4),
        "roc_auc": round(roc_auc_score(y_test, y_proba), 4),
    }
    print(f"{name}: {metrics}")
    return metrics, y_pred, y_proba

candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=80, max_depth=2, random_state=RANDOM_STATE),
}

results = {}
for name, model in candidates.items():
    model.fit(X_train_bal, y_train_bal)
    metrics, _, _ = evaluate_model(name, model, X_test_scaled, y_test)
    results[name] = metrics

results

## 7. Hyperparameter Tuning (Best Model)

In [ ]:
best_name = max(results, key=lambda n: results[n]["f1_score"])
print("Best baseline model:", best_name)

# Load the pre-tuned model produced by scripts/03_train_models.py
best_model = joblib.load(os.path.join(MODELS_DIR, "fraud_detection_model.joblib"))
with open(os.path.join(MODELS_DIR, "model_metrics.json")) as f:
    saved_metrics = json.load(f)

print("Tuned model:", saved_metrics["best_model_name"])
print("Best parameters:", saved_metrics["best_params"])
print("Final test metrics:", saved_metrics["final_metrics"])

## 8. Model Evaluation (Precision, Recall, F1, ROC-AUC)

In [ ]:
y_pred = best_model.predict(X_test_scaled)
y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred, target_names=["Legitimate", "Fraud"]))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=["Legitimate", "Fraud"], yticklabels=["Legitimate", "Fraud"])
ax.set_title("Confusion Matrix")
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="#E63946", label=f"ROC AUC = {saved_metrics['final_metrics']['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve")
ax.legend()
plt.show()

## 9. Fraud Pattern & Feature Importance Analysis

In [ ]:
if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=feature_cols)
    top = importances.sort_values(ascending=False).head(12)
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(top.index[::-1], top.values[::-1], color="#2E86AB")
    ax.set_title("Top Features Driving Fraud Predictions")
    plt.show()
    display(top)

In [ ]:
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

print("Median amount — fraud:", round(fraud["Amount"].median(), 2),
      " legitimate:", round(legit["Amount"].median(), 2))

fraud_hour_counts = fraud["Hour"].value_counts().sort_index()
print("\nFraud transactions by hour of day:")
print(fraud_hour_counts)

## 10. Business Insights & Recommendations

- Fraudulent transactions cluster at a lower median amount than legitimate ones,
  consistent with "card testing" behavior where small charges are used to validate
  stolen card details before a larger purchase is attempted.
- A concentrated subset of anonymized features (see feature importance above)
  carries most of the predictive signal, meaning a lean, fast-scoring model is
  practical for real-time transaction screening.
- The tuned model favors recall for the fraud class, catching a substantially
  higher share of fraudulent transactions than an untuned baseline, at an
  acceptable cost in false positives — a reasonable trade-off given that missed
  fraud is typically far more expensive than a manual review of a flagged
  legitimate transaction.
- Recommended next steps: deploy the model as a first-pass screening layer ahead
  of manual review, monitor precision/recall drift monthly as spending patterns
  evolve, and periodically retrain on newly confirmed fraud cases.